In [12]:
import os
os.environ['KAGGLE_USERNAME'] = 'NISHAA'
os.environ['KAGGLE_KEY'] = '6c9c7ac2c7edc579c206c0d6347dde71'

In [13]:
!pip install kaggle

In [14]:
!kaggle datasets download -d 'abdelazizsami/predictive-maintenance-dataset'

Dataset URL: https://www.kaggle.com/datasets/abdelazizsami/predictive-maintenance-dataset
License(s): apache-2.0
100% 136k/136k [00:00<00:00, 59.6MB/s]



In [15]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

import joblib

In [16]:
df = pd.read_csv('/content/predictive-maintenance-dataset.zip')

In [17]:
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [18]:
df.tail()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
9995,9996,M24855,M,298.8,308.4,1604,29.5,14,0,0,0,0,0,0
9996,9997,H39410,H,298.9,308.4,1632,31.8,17,0,0,0,0,0,0
9997,9998,M24857,M,299.0,308.6,1645,33.4,22,0,0,0,0,0,0
9998,9999,H39412,H,299.0,308.7,1408,48.5,25,0,0,0,0,0,0
9999,10000,M24859,M,299.0,308.7,1500,40.2,30,0,0,0,0,0,0


In [19]:
df.describe()

,UDI,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
count,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,5000.50000,300.004930,310.005560,1538.776100,39.986910,107.951000,0.033900,0.004600,0.011500,0.009500,0.009800,0.00190
std,2886.89568,2.000259,1.483734,179.284096,9.968934,63.654147,0.180981,0.067671,0.106625,0.097009,0.098514,0.04355
min,1.00000,295.300000,305.700000,1168.000000,3.800000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,2500.75000,298.300000,308.800000,1423.000000,33.200000,53.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
50%,5000.50000,300.100000,310.100000,1503.000000,40.100000,108.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
75%,7500.25000,301.500000,311.100000,1612.000000,46.800000,162.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
max,10000.00000,304.500000,313.800000,2886.000000,76.600000,253.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9)

In [21]:
df.shape

(10000, 14)

In [24]:
df.dtypes

,0
UDI,int64
Product ID,object
Type,object
Air temperature [K],float64
Process temperature [K],float64
Rotational speed [rpm],int64
Torque [Nm],float64
Tool wear [min],int64
Machine failure,int64
TWF,int64


In [22]:
df.isnull().sum()

,0
UDI,0
Product ID,0
Type,0
Air temperature [K],0
Process temperature [K],0
Rotational speed [rpm],0
Torque [Nm],0
Tool wear [min],0
Machine failure,0
TWF,0


In [23]:
df.duplicated().sum()

np.int64(0)

In [27]:
le = LabelEncoder()

df["Product ID"] = le.fit_transform(df["Product ID"])
df["Type"] = le.fit_transform(df["Type"])

In [28]:
df["Temperature Difference"] = df["Process temperature [K]"] - df["Air temperature [K]"]

df["Power"] = df["Torque [Nm]"] * df["Rotational speed [rpm]"]

In [45]:
df.drop( ["TWF", "HDF", "PWF", "OSF"], axis=1, inplace=True)

In [46]:
X = df.drop("Machine failure", axis=1)
y = df["Machine failure"]

In [47]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [48]:
dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [49]:
y_pred_dt = dt.predict(X_test)

In [60]:
accuracy_score(y_test, y_pred_dt)
print("Accuracy Score: ", accuracy_score(y_test, y_pred_dt))

Accuracy Score:  0.9805


In [51]:
cm_dt = confusion_matrix(y_test, y_pred_dt)

print(cm_dt)

[[1915   24]
 [  15   46]]


In [52]:
print(classification_report(y_test, y_pred_dt))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1939
           1       0.66      0.75      0.70        61

    accuracy                           0.98      2000
   macro avg       0.82      0.87      0.85      2000
weighted avg       0.98      0.98      0.98      2000



In [53]:
rf = RandomForestClassifier(random_state=42)

In [54]:
rf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [55]:
y_pred_rf = rf.predict(X_test)

In [59]:
accuracy_score(y_test, y_pred_rf)
print("Accuracy Score: ", accuracy_score(y_test, y_pred_rf))

Accuracy Score:  0.9905


In [57]:
cm_rf = confusion_matrix(y_test, y_pred_rf)

print(cm_rf)

[[1937    2]
 [  17   44]]


In [58]:
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.99      1.00      1.00      1939
           1       0.96      0.72      0.82        61

    accuracy                           0.99      2000
   macro avg       0.97      0.86      0.91      2000
weighted avg       0.99      0.99      0.99      2000



In [61]:
results = pd.DataFrame({
    "Algorithm": ["Decision Tree", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_rf)
    ]
})

print(results)

       Algorithm  Accuracy
0  Decision Tree    0.9805
1  Random Forest    0.9905


In [62]:
import joblib

joblib.dump(rf, "equipment_health_model.pkl")

['equipment_health_model.pkl']

In [63]:
from google.colab import files

files.download("equipment_health_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>